# LLM-based Feature Engineering using Ollama (Fully Free)

This notebook builds a fraud detection pipeline using:
- Ollama (local LLM)
- Behavioral feature generation
- DistilBERT model
- Iterative loop (semi-automated)


## ✅ STEP 0: Install Ollama (RUN IN TERMINAL, NOT NOTEBOOK)

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3
ollama run llama3
```

Keep Ollama running in background.

In [1]:
# STEP 1: Install dependencies
!pip install pandas numpy scikit-learn transformers datasets tqdm requests

In [2]:
# STEP 2: Load dataset
import pandas as pd

df = pd.read_csv('../../transactions/card_transaction.v1.csv')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [3]:
# STEP 3: Create user-level fraud label
fraud_users = df.groupby('User')['Is Fraud?'].max().reset_index()
fraud_users.columns = ['User', 'User_Fraud_Label']

df = df.merge(fraud_users, on='User')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,User_Fraud_Label
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,Yes
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,Yes
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,Yes
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,Yes
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,Yes


In [4]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()['response']

In [5]:
# STEP 5: Generate features using LLM

schema = list(df.columns)

prompt = f"""
Dataset columns: {schema}

Generate fraud detection features.

IMPORTANT:
- "computation" MUST be valid pandas code
- Use dataframe name: df
- Do NOT explain anything

Example:
df['avg_amount'] = df.groupby('User')['Amount'].transform('mean')

STRICT JSON ONLY:
[
  {{
    "feature_name": "",
    "description": "",
    "type": "numerical/categorical",
    "computation": "VALID PYTHON CODE",
    "intuition": ""
  }}
]
"""

llm_output = call_llm(prompt)
print(llm_output)

Here is the list of fraud detection features in JSON format:

[
  {
    "feature_name": "avg_amount_per_user",
    "description": "",
    "type": "numerical",
    "computation": "df['avg_amount_per_user'] = df.groupby('User')['Amount'].transform('mean')",
    "intuition": ""
  },
  {
    "feature_name": "total_transactions_per_user",
    "description": "",
    "type": "numerical",
    "computation": "df['total_transactions_per_user'] = df.groupby('User')['Time'].count()",
    "intuition": ""
  },
  {
    "feature_name": "max_amount_per_user",
    "description": "",
    "type": "numerical",
    "computation": "df['max_amount_per_user'] = df.groupby('User')['Amount'].transform('max')",
    "intuition": ""
  },
  {
    "feature_name": "min_amount_per_user",
    "description": "",
    "type": "numerical",
    "computation": "df['min_amount_per_user'] = df.groupby('User')['Amount'].transform('min')",
    "intuition": ""
  },
  {
    "feature_name": "merchant_frequency_per_user",
    "descri

In [ ]:
# STEP 6: Example feature implementation (manual first iteration)

#print(df['Amount'].dtype)
#print(df['Amount'].head(10))
df['Amount'] = df['Amount'].replace('[\$,]', '', regex=True).astype(float)

# df['txn_count'] = df.groupby('User')['Amount'].transform('count')
# df['avg_amount'] = df.groupby('User')['Amount'].transform('mean')
# df['amount_std'] = df.groupby('User')['Amount'].transform('std')
# df['unique_merchants'] = df.groupby('User')['Merchant Name'].transform('nunique')

# df.head()

In [18]:
df['unique_merchants_per_user'] = df.groupby('User')['Merchant Name'].transform('nunique')

# Fix Use Chip
if df['Use Chip'].dtype == 'object':
    df['Use Chip'] = df['Use Chip'].map({'Yes': 1, 'No': 0}).fillna(0)
    
df['merchant_frequency_per_user'] = df.groupby('User')['Merchant Name'].transform('count')

In [19]:
import json
import re

def extract_json(llm_output):
    try:
        # Extract JSON block
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    
def apply_features_exec(df, features):
    for feature in features:
        code = feature.get("computation", "")

        print(f"\nExecuting:\n{code}")

        try:
            exec(code)
        except Exception as e:
            print(f"❌ Error: {e}")

    return df

# Parse LLM output
features = extract_json(llm_output)

# Apply features
df = apply_features_exec(df, features)

df.head()


Executing:
df['avg_amount_per_user'] = df.groupby('User')['Amount'].transform('mean')

Executing:
df['total_transactions_per_user'] = df.groupby('User')['Time'].count()

Executing:
df['max_amount_per_user'] = df.groupby('User')['Amount'].transform('max')

Executing:
df['min_amount_per_user'] = df.groupby('User')['Amount'].transform('min')

Executing:
df['merchant_frequency_per_user'] = df.groupby(['User', 'Merchant Name']).size().groupby('User').transform('sum')
❌ Error: incompatible index of inserted column with frame index

Executing:
df['chip_usage_percentage'] = df.groupby('User')['Use Chip'].mean()

Executing:
df['fraudulent_transactions_per_user'] = df.groupby('User')['Is Fraud?'].sum()


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,...,Is Fraud?,User_Fraud_Label,total_transactions_per_user,max_amount_per_user,min_amount_per_user,fraudulent_transactions_per_user,avg_amount_per_user,unique_merchants_per_user,chip_usage_percentage,merchant_frequency_per_user
0,0,0,2002,9,1,06:21,134.09,0.0,3527213246127876953,La Verne,...,No,Yes,19963.0,1409.4,-499.0,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,81.299989,552,0.0,19963
1,0,0,2002,9,1,06:42,38.48,0.0,-727612092139916043,Monterey Park,...,No,Yes,8919.0,1409.4,-499.0,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,81.299989,552,0.0,19963
2,0,0,2002,9,2,06:22,120.34,0.0,-727612092139916043,Monterey Park,...,No,Yes,41978.0,1409.4,-499.0,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,81.299989,552,0.0,19963
3,0,0,2002,9,2,17:45,128.95,0.0,3414527459579106770,Monterey Park,...,No,Yes,10117.0,1409.4,-499.0,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,81.299989,552,0.0,19963
4,0,0,2002,9,3,06:23,104.71,0.0,5817218446178736267,La Verne,...,No,Yes,18542.0,1409.4,-499.0,NoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNoNo...,81.299989,552,0.0,19963


In [20]:
# Select only numeric features (important)
feature_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove label + unwanted columns
feature_cols = [col for col in feature_cols if col not in ['Is Fraud?', 'User_Fraud_Label']]

print("Using features:", feature_cols)

X = df[feature_cols]
y = df['User_Fraud_Label']

Using features: ['User', 'Card', 'Year', 'Month', 'Day', 'Amount', 'Use Chip', 'Merchant Name', 'Zip', 'MCC', 'total_transactions_per_user', 'max_amount_per_user', 'min_amount_per_user', 'avg_amount_per_user', 'unique_merchants_per_user', 'chip_usage_percentage', 'merchant_frequency_per_user']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)



In [ ]:

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))

In [ ]:
# # STEP 7: Convert features to text for DistilBERT

# def row_to_text(row):
#     return f"User behavior: avg {row['avg_amount']}, count {row['txn_count']}, std {row['amount_std']}"

# df['text'] = df.apply(row_to_text, axis=1)
# df[['text', 'User_Fraud_Label']].head()

In [ ]:
# # STEP 8: Train DistilBERT
# from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
# from datasets import Dataset

# dataset = Dataset.from_pandas(df[['text', 'User_Fraud_Label']])

# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# def tokenize(example):
#     return tokenizer(example['text'], truncation=True, padding='max_length')

# dataset = dataset.map(tokenize, batched=True)
# dataset = dataset.train_test_split(test_size=0.2)

# model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=2,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     evaluation_strategy='epoch'
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset['train'],
#     eval_dataset=dataset['test']
# )

# trainer.train()

In [ ]:
# # STEP 9: Evaluate
# metrics = trainer.evaluate()
# print(metrics)

## 🔁 STEP 10: Iteration (Manual + Guided)

If performance is low:
- Re-run STEP 5 with improved prompt
- Add/remove features
- Retrain model

Optional: Add feedback into prompt like:

Model struggling with temporal patterns → generate time-based features
